# Derived distributions

_Probability & Statistics_

**Push a random variable through a function. Here is how its spread of values gets reshaped.**

You know the distribution of $X$. You build a new variable $Y = g(X)$.

     What is the distribution of $Y$?

---

This notebook builds a derived distribution one step at a time: write down the change-of-variables formula, check it against simulation bin by bin, confirm the mean, then picture how squaring reshapes the spread. Run each cell and read the note above it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy + SciPy

### Step 1 — Write down the derived density

Start with `X ~ Uniform[0,1]` and define `Y = X^2`. The change-of-variables formula gives the density of `Y` as `f_Y(y) = 1/(2*sqrt(y))`. Notice it blows up near `y = 0`: squaring piles probability mass toward zero, since many small `x` values land in a tiny range of `y`. We also draw a million samples of `Y` to test the formula against.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Derived dist: X ~ Uniform[0,1], Y = X^2. Formula: f_Y(y) = 1/(2*sqrt(y)).
f_Y = lambda y: 1.0 / (2.0 * np.sqrt(y))

# Monte-Carlo: simulate Y by squaring uniform draws.
x = rng.random(2_000_000)
y = x ** 2

### Step 2 — Compare density bin by bin

To validate the formula we pick a few bins (starting above `0.04` to dodge the spike at `y = 0`). For each bin, the **empirical** density is the fraction of samples in it divided by its width; the **theoretical** density is the formula integrated over the bin, then divided by width to get an average. The two columns should line up closely.

In [ ]:
from scipy import integrate

edges = np.array([0.04, 0.09, 0.16, 0.25])   # avoid y=0 spike

for a, b in zip(edges[:-1], edges[1:]):
    width = b - a
    emp = np.mean((y >= a) & (y < b)) / width        # empirical density
    theo_mass, _ = integrate.quad(f_Y, a, b)
    theo = theo_mass / width                         # theoretical avg density
    print(f"y in [{a:.2f},{b:.2f}): MC={emp:.3f}  formula={theo:.3f}")

### Step 3 — Check the mean of Y

As a final sanity check, the expected value `E[Y] = E[X^2]` for `X ~ Uniform[0,1]` equals `1/3`. The sample mean of our simulated `Y` should land right on it.

In [ ]:
sample_mean = y.mean()

print("E[Y]  MC =", round(sample_mean, 4), " formula =", round(1/3, 4))

## Visualize the data & results

_Does the change-of-variables formula f_Y(y) = 1/(2 sqrt(y)) match a simulated histogram of Y = X^2?_

### Step 1 — Simulate Y and evaluate the formula curve

We regenerate `Y = X^2` from a fresh seeded stream, then evaluate the formula `1/(2*sqrt(y))` at a set of bin centers. These formula heights are what the simulated histogram should match.

In [ ]:
import numpy as np

# X ~ Uniform[0,1], Y = X^2.  Formula density: f_Y(y) = 1/(2 sqrt(y)).
rng = np.random.default_rng(0)
y_sim = rng.random(2_000_000) ** 2

# Formula curve evaluated at bin centers.
centers = np.array([0.065, 0.125, 0.205, 0.305, 0.425, 0.565, 0.725])
formula = 1.0 / (2.0 * np.sqrt(centers))

### Step 2 — Histogram the simulation at matching bins

We bin the simulated `Y` with `density=True` so the bar heights are densities directly comparable to the formula. The bin edges are chosen so each `centers` value sits inside one bin.

In [ ]:
# Monte-Carlo histogram heights at the same bins.
edges = np.array([0.04, 0.09, 0.16, 0.25, 0.36, 0.49, 0.64, 0.81])
mc, _ = np.histogram(y_sim, bins=edges, density=True)

### Step 3 — Plot formula vs simulation, and the density shape

Left overlays the formula curve on the Monte-Carlo histogram heights — they should track each other. Right shows the formula density evaluated at a row of sample `y` values, making the steep rise toward `y = 0` plain: squaring crowds the spread near zero.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Chart 1: formula curve vs Monte-Carlo histogram heights.
ax1.plot(centers, formula, '-o', color='#ffb454', label='formula 1/(2 sqrt(y))')
ax1.plot(centers, mc, '-o', color='#4ea1ff', label='Monte-Carlo histogram')
ax1.set_xlabel('y')
ax1.set_ylabel('density f_Y(y)')
ax1.legend()
ax1.set_title('Derived density of Y = X^2: formula vs simulation')

# Chart 2: formula density height at sample y values (squaring piles mass near 0).
ys = np.array([0.04, 0.09, 0.16, 0.25, 0.36, 0.49, 0.64, 0.81])
heights = 1.0 / (2.0 * np.sqrt(ys))
ax2.bar([str(v) for v in ys], heights, color='#7ee787')
ax2.set_title('Density height at sample y values (formula)')

plt.tight_layout()
plt.show()